# FV 02 — Point-in-time fundamentals panel (monthly, 2016–2025)

Protocol: [`discussions/Fair_Value/002`](../../discussions/Fair_Value/002_claude_fv02_panel.md). Data build plus checks; still no signal and no returns.

**Inputs**
- FV 01 EDGAR files.
- Alpaca 1Day bars: adjusted (`bars/1Day`) and **raw** (`bars/1Day_raw`). Run `python tools/prices/daily_download.py --source alpaca --adjustment raw` first.

**Steps**
1. Filing snapshots.
2. Month-end prices.
3. Ticker → company mapping over time.
4. Monthly panel with market cap and EV.
5. Exits table.

**Checks**
- A. Apple values against its reported figures.
- B. Universe per year after filters.
- C. DCF item coverage on the priced universe.
- D. Price coverage of companies that filed a Form 25 (exchange delisting).
- E. Mapping sanity: market cap vs public float, ambiguous tickers.
- F. Exits by type and year.

Heavy steps are cached under `<EQUITY_TRADING_DIR>/EquityTrading/fair_value/`. Delete a file there to rebuild it.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parents[1] / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from equity_trading import edgar as E, fv_panel as F

OUT = Path("02"); OUT.mkdir(exist_ok=True)
FV = F.fv_dir()
e = E.edgar_dir()
print("fv_panel", F.FV_PANEL_VERSION, "| edgar", E.EDGAR_VERSION, "|", FV)
companies = pd.read_parquet(e / "companies.parquet", columns=["cik", "name", "sic"])
filings = pd.read_parquet(e / "filings.parquet", columns=["cik", "accn", "form", "filing_date", "accepted"])
filings = filings[filings["filing_date"] <= "2025-12-31"]
status = E.company_status(filings.assign(form=filings["form"].astype(str)))

fv_panel 0.1.1 | edgar 0.1.2 | D:\MarketData\EquityTrading\fair_value


## 1. Filing snapshots (cached; first build roughly 20–60 min with 3 workers)

In [2]:
summary = F.build_snapshots(workers=3)
if len(summary):
    display(summary.tail())
snap = F.load_snapshots()
print(f"{len(snap):,} snapshots, {snap['cik'].nunique():,} CIKs, accepted {snap['accepted'].min():%Y-%m-%d} .. {snap['accepted'].max():%Y-%m-%d}")

snapshots built by fv_panel 0.1.0; rebuilding with 0.1.1
42 of 42 facts parts to process, 3 workers


[Parallel(n_jobs=3)]: Using backend LokyBackend with 3 concurrent workers.
[Parallel(n_jobs=3)]: Done  12 tasks      | elapsed:  4.7min
[Parallel(n_jobs=3)]: Done  42 out of  42 | elapsed:  9.7min finished


,part,ciks,snapshots,seconds
37,part-00037.parquet,154,1136,6.4
38,part-00038.parquet,199,856,5.9
39,part-00039.parquet,175,324,3.2
40,part-00040.parquet,0,0,0.4
41,part-00041.parquet,0,0,0.4


696,589 snapshots, 14,756 CIKs, accepted 1993-08-13 .. 2025-12-31


### A. Apple check (CIK 320193) against reported figures

In [3]:
aapl = snap[snap["cik"] == 320193].set_index("period_end").sort_index()
cols = ["form", "accepted", "ttm_revenue", "ttm_net_income", "ttm_cfo", "ttm_capex", "fy0_revenue", "fy1_revenue",
        "cash", "debt", "shares"]
display(aapl.loc["2022-09-01":"2024-01-31", cols].assign(**{c: (lambda x, c=c: x[c] / 1e9) for c in cols[2:]}))
# Reported (USD bn): FY2023 (Sep-23) revenue 383.285, FY2022 394.328; TTM to Dec-23 = 383.285 + 119.575 - 117.154 = 385.706
checks = [("FY2023 revenue", "2023-09-30", "ttm_revenue", 383.285),
          ("FY2022 revenue (fy1 at the FY2023 10-K)", "2023-09-30", "fy1_revenue", 394.328),
          ("TTM revenue to Dec-2023", "2023-12-30", "ttm_revenue", 385.706)]
for name, pe, col, exp in checks:
    rows = aapl.loc[aapl.index == pe, col]
    got = rows.iloc[-1] / 1e9 if len(rows) else np.nan
    print(f"{name:42s} got {got:9.3f}  expected {exp:9.3f}  {'OK' if abs(got - exp) < 0.01 else 'MISMATCH'}")

,form,accepted,ttm_revenue,ttm_net_income,ttm_cfo,ttm_capex,fy0_revenue,fy1_revenue,cash,debt,shares
period_end,,,,,,,,,,,
2022-09-24,10-K,2022-10-27 22:01:14,394.328,99.803,122.151,10.708,394.328,365.817,23.646,120.069,15.908118
2022-12-31,10-Q,2023-02-02 23:01:30,387.537,95.171,109.190,11.692,394.328,365.817,20.535,111.110,15.821946
2023-04-01,10-Q,2023-05-04 22:03:52,385.095,94.321,109.584,12.094,394.328,365.817,24.687,109.615,15.728702
2023-07-01,10-Q,2023-08-03 22:04:43,383.933,94.760,113.072,12.085,394.328,365.817,28.408,109.280,15.634232
2023-09-30,10-K,2023-11-02 22:08:27,383.285,96.995,110.543,10.959,383.285,394.328,29.965,111.088,15.552752
2023-12-30,10-Q,2024-02-01 23:03:38,385.706,100.913,116.433,9.564,383.285,394.328,40.760,108.040,15.441881


FY2023 revenue                             got   383.285  expected   383.285  OK
FY2022 revenue (fy1 at the FY2023 10-K)    got   394.328  expected   394.328  OK
TTM revenue to Dec-2023                    got   385.706  expected   385.706  OK


## 2. Month-end prices from Alpaca (raw + adjusted)

In [4]:
mp_path = FV / "month_end_prices.parquet"
syms = pd.read_csv(e / "price_symbols.csv")["alpaca"].dropna().unique().tolist()
if mp_path.exists():
    mp = pd.read_parquet(mp_path)
else:
    mp = F.month_end_prices(syms, workers=4)
print(f"{mp['symbol'].nunique():,} symbols with raw bars, {len(mp):,} symbol-months")
if mp["close_adj"].isna().mean() > 0.05:
    print(f"warning: {mp['close_adj'].isna().mean():.1%} of months have no adjusted close (adjusted cache incomplete?)")

10,818 symbols with raw bars, 741,938 symbol-months


## 3. Ticker → company mapping and 4. monthly panel

In [5]:
th = pd.read_parquet(e / "ticker_history.parquet")
win = F.symbol_windows(th, status)
assigned = F.assign_symbols(win, mp)
print(f"assigned {len(assigned):,} CIK-months, {assigned['cik'].nunique():,} CIKs; "
      f"ambiguous symbol-months dropped: {assigned.attrs['ambiguous_dropped']:,}")
months = pd.date_range("2016-01-31", "2025-12-31", freq="ME")
panel = F.monthly_panel(snap, assigned, companies, months=months, prices=mp)
panel.to_parquet(FV / "panel_monthly.parquet", index=False)
print(f"panel: {len(panel):,} rows, {panel['cik'].nunique():,} CIKs")

assigned 532,211 CIK-months, 8,667 CIKs; ambiguous symbol-months dropped: 14,436
panel: 448,551 rows, 7,201 CIKs


### B. Universe per year (December of each year)

In [6]:
dec = panel[panel["month_end"].dt.month == 12].copy()
dec["year"] = dec["month_end"].dt.year
f_ok = (~dec["is_financial"]) & (~dec["is_blank_check"]) & (dec["mcap"] > 0)
f_map = f_ok & ~dec["mcap_suspect"]
f_dcf = f_map & dec["ttm_revenue"].notna() & dec["ttm_fcf"].notna() & dec["fy4_fcf"].notna() & (dec["ev"] > 0)
u = pd.DataFrame({"priced": dec.groupby("year")["cik"].nunique(),
                  "non_fin_non_spac": dec[f_ok].groupby("year")["cik"].nunique(),
                  "mapping_ok": dec[f_map].groupby("year")["cik"].nunique(),
                  "dcf_ready_5y": dec[f_dcf].groupby("year")["cik"].nunique()})
bins = [-np.inf, 3e8, 2e9, 1e10, 2e11, np.inf]
labels = ["<300M", "300M-2B", "2B-10B", "10B-200B", ">200B"]
evb = dec[f_dcf].assign(ev_bucket=pd.cut(dec.loc[f_dcf, "ev"], bins, labels=labels))
u = u.join(evb.groupby(["year", "ev_bucket"], observed=False)["cik"].nunique().unstack())
u.to_csv(OUT / "universe_by_year.csv"); display(u)

,priced,non_fin_non_spac,mapping_ok,dcf_ready_5y,<300M,300M-2B,2B-10B,10B-200B,>200B
year,,,,,,,,,
2016,3911,2888,2873,2122,618,652,522,315,15
2017,3874,2867,2724,2056,561,624,514,340,17
2018,3918,2901,2785,2068,594,607,519,328,20
2019,3873,2835,2681,2075,527,591,560,377,20
2020,3841,2752,2566,1961,497,549,493,394,28
2021,4142,2941,2813,1882,459,451,500,431,41
2022,3506,2676,2511,1622,283,438,478,394,29
2023,3384,2609,2355,1607,271,399,481,418,38
2024,3351,2564,2310,1674,264,426,508,436,40


### C. DCF item coverage on the priced, non-financial universe

In [7]:
items = ["ttm_revenue", "ttm_operating_income", "ttm_net_income", "ttm_cfo", "ttm_capex", "ttm_d_and_a",
         "ttm_interest_expense", "ttm_income_tax", "cash", "debt", "equity", "shares", "fy4_revenue", "fy4_fcf"]
d = dec[f_ok]
cov = d.groupby("year")[items].agg(lambda x: x.notna().mean())
cov["debt_reported"] = d.groupby("year")["debt_reported"].mean()
cov["shares_from_dei"] = d.groupby("year")["shares_source"].agg(lambda x: (x == "dei").mean())
cov.round(3).to_csv(OUT / "item_coverage_priced.csv"); display(cov.round(2))

,ttm_revenue,ttm_operating_income,ttm_net_income,ttm_cfo,ttm_capex,ttm_d_and_a,ttm_interest_expense,ttm_income_tax,cash,debt,equity,shares,fy4_revenue,fy4_fcf,debt_reported,shares_from_dei
year,,,,,,,,,,,,,,,,
2016,0.85,0.88,0.95,0.95,0.88,0.91,0.73,0.82,0.99,1.0,0.97,1.0,0.76,0.83,0.650277,0.96
2017,0.84,0.87,0.95,0.95,0.86,0.91,0.72,0.80,0.99,1.0,0.97,1.0,0.77,0.84,0.657133,0.95
2018,0.83,0.87,0.94,0.93,0.85,0.89,0.71,0.79,0.99,1.0,0.98,1.0,0.77,0.84,0.652878,0.95
2019,0.87,0.88,0.95,0.95,0.87,0.90,0.71,0.79,0.99,1.0,0.98,1.0,0.77,0.84,0.640212,0.95
2020,0.85,0.87,0.94,0.94,0.84,0.89,0.69,0.77,0.99,1.0,0.98,1.0,0.75,0.82,0.643169,0.95
2021,0.78,0.81,0.87,0.87,0.77,0.81,0.64,0.70,0.99,1.0,0.98,1.0,0.70,0.74,0.623937,0.92
2022,0.84,0.89,0.95,0.95,0.85,0.88,0.70,0.79,0.99,1.0,0.98,1.0,0.68,0.71,0.641256,0.90
2023,0.84,0.89,0.96,0.97,0.84,0.89,0.70,0.80,0.99,1.0,0.98,1.0,0.69,0.73,0.63281,0.91
2024,0.84,0.89,0.96,0.96,0.82,0.89,0.71,0.80,0.99,1.0,0.99,1.0,0.74,0.79,0.629095,0.91


### D. Price coverage of exchange-delisted companies (Form 25 / 25-NSE filers, 2016–2025)

In [8]:
spac = set(companies.loc[companies["sic"].astype(str) == "6770", "cik"])
f25 = filings[filings["form"].astype(str).isin(["25", "25-NSE"]) & (filings["filing_date"] >= "2016-03-01")]
f25 = f25.sort_values("filing_date").drop_duplicates("cik", keep="last")[["cik", "filing_date"]]
last_px = assigned.groupby("cik")["last_date"].max()
f25["last_price"] = f25["cik"].map(last_px)
f25["covered"] = f25["last_price"].notna() & ((f25["filing_date"] - f25["last_price"]).dt.days <= 120)
f25["periodic_filer"] = f25["cik"].isin(status["cik"])
f25["blank_check"] = f25["cik"].isin(spac)
d25 = f25[f25["periodic_filer"]]
t = (d25.groupby([d25["filing_date"].dt.year, d25["blank_check"].map({True: "spac", False: "operating"})])["covered"]
     .agg(["mean", "count"]).unstack())
t.round(3).to_csv(OUT / "form25_price_coverage.csv"); display(t.round(3))
op = d25[~d25["blank_check"]]
print(f"operating-company Form 25 filers: {len(op):,}; priced up to within 120 days of the form: {op['covered'].mean():.1%}")

mean            count      
blank_check operating   spac operating  spac
filing_date                                 
2016            0.836    NaN     189.0   NaN
2017            0.842  1.000     234.0   1.0
2018            0.855  1.000     179.0   1.0
2019            0.923  0.000     194.0   1.0
2020            0.871    NaN     155.0   NaN
2021            0.763  0.600     219.0   5.0
2022            0.737  0.545     171.0  11.0
2023            0.568  0.000     199.0  13.0
2024            0.554  0.222     202.0   9.0
2025            0.624  0.000     250.0   7.0

operating-company Form 25 filers: 1,992; priced up to within 120 days of the form: 75.1%


### E. Mapping sanity

In [9]:
pf = panel[panel["shares_to_float"].notna() & ~panel["is_financial"] & ~panel["is_blank_check"]]
print(f"rows with public float and a float-date price: {len(pf):,}; shares / float shares outside [0.8, 50]: {pf['mcap_suspect'].mean():.1%}")
display(pf["shares_to_float"].quantile([0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).round(2).to_frame("shares / float shares"))
by_year = pf.groupby(pf["month_end"].dt.year)["mcap_suspect"].mean()
display(by_year.round(3).to_frame("suspect share"))
worst = (pf[pf["mcap_suspect"]].groupby(["cik", "name", "symbol"])["shares_to_float"].median()
         .sort_values().reset_index())
display(pd.concat([worst.head(15), worst.tail(15)]))
worst.to_csv(OUT / "mcap_float_suspects.csv", index=False)
print("shares source:", panel["shares_source"].value_counts(normalize=True).round(3).to_dict())

rows with public float and a float-date price: 269,613; shares / float shares outside [0.8, 50]: 6.7%


,shares / float shares
0.01,0.00
0.05,0.51
0.25,1.01
0.50,1.19
0.75,1.66
0.95,4.06
0.99,16.15


,suspect share
month_end,
2016,0.093
2017,0.048
2018,0.042
2019,0.051
2020,0.065
2021,0.058
2022,0.058
2023,0.086
2024,0.095


,cik,name,symbol,shares_to_float
0,29915,DOW CHEMICAL CO /DE/,DOW,8.908602e-08
1,30554,"EIDP, Inc.",DD,9.010000e-08
2,1425205,"IOVANCE BIOTHERAPEUTICS, INC.",IOVA,1.112838e-07
3,1842939,"Longevity Health Holdings, Inc.",CTCX,5.361000e-07
4,1840425,"OSR Health, Inc.",BLAC,7.003094e-07
5,871763,ManpowerGroup Inc.,MAN,9.612942e-07
6,717605,HEXCEL CORP /DE/,HXL,9.838983e-07
7,16040,CABOT CORP,CBT,9.929910e-07
8,1707092,Delphi Technologies PLC,DLPH,9.934513e-07
9,1136869,"ZIMMER BIOMET HOLDINGS, INC.",ZBH,1.006325e-06


shares source: {'dei': 0.9, 'diluted_avg': 0.053, 'none': 0.039, 'gaap': 0.007}


### F. Exits (CIKs whose price stops before 2025-12)

In [10]:
ex = F.exits(filings, mp, assigned)
ex["blank_check"] = ex["cik"].isin(spac)
ex.to_parquet(FV / "exits.parquet", index=False)
t = ex.groupby([ex["last_date"].dt.year, ex["blank_check"].map({True: "spac", False: "operating"}), "exit_type"]).size().unstack([1, 2], fill_value=0)
t.to_csv(OUT / "exits_by_year.csv"); display(t)

blank_check operating                    spac               
exit_type    acquired delisted_other acquired delisted_other
last_date                                                   
2016              206            147        2              2
2017              192            105        5              1
2018              191            118        2              2
2019              181            163        6              4
2020              134            394        6              2
2021              142            432       31             10
2022              166            806       14            180
2023              136            147        0             15
2024              119            140        0              4
2025              143            129        0              1

### G. Survivorship where it matters: companies that would have entered the DCF universe

D and F count every filer, including SPACs without SIC 6770, shells and micro caps that never pass the DCF filters. What matters is narrower:
- take companies that had, within 24 months before their exit, a filing with a 5-year FCF history, revenue > 0, not financial, not a SPAC, and public float ≥ $100M;
- of those, how many filed a Form 25, and how many are priced up to it?

This uses fundamentals and public float only, so an unpriced company can still qualify.

In [ ]:
sic_num = pd.to_numeric(companies.set_index("cik")["sic"], errors="coerce")
bad_sic = set(sic_num[sic_num.between(6000, 6799)].index)
elig = snap[snap["fy4_fcf"].notna() & (snap["ttm_revenue"] > 0) & (snap["public_float"] >= 1e8)
            & ~snap["cik"].isin(bad_sic)][["cik", "accepted", "public_float"]]
op25 = d25[~d25["blank_check"]].copy()
m = op25.merge(elig, on="cik")
m = m[(m["accepted"] <= m["filing_date"]) & (m["accepted"] >= m["filing_date"] - pd.Timedelta(days=730))]
last_float = m.sort_values("accepted").groupby("cik")["public_float"].last()
g25 = op25[op25["cik"].isin(last_float.index)].assign(public_float=lambda x: x["cik"].map(last_float))
g = g25.groupby(g25["filing_date"].dt.year)["covered"].agg(["mean", "count"])
g.round(3).to_csv(OUT / "form25_coverage_dcf_eligible.csv"); display(g.round(3))
print(f"DCF-eligible Form 25 filers: {len(g25):,}; priced up to within 120 days of the form: {g25['covered'].mean():.1%}")
fb = pd.cut(g25["public_float"], [1e8, 3e8, 1e9, 1e10, np.inf], labels=["100-300M", "300M-1B", "1-10B", ">10B"])
display(g25.groupby(fb, observed=False)["covered"].agg(["mean", "count"]).round(3))
# uncovered examples, with their symbol candidates and what Alpaca has for them
unc = g25[~g25["covered"]].merge(companies[["cik", "name"]], on="cik")
cand = th.groupby("cik")["symbol"].agg(lambda s: "|".join(sorted(set(s))))
span = mp.groupby("symbol")["last_date"].agg(["min", "max"])
def alp(syms):
    out = []
    for s in str(syms).split("|"):
        s2 = s.replace("-", ".")
        if s2 in span.index:
            out.append(f"{s2}:{span.at[s2, 'min']:%Y-%m}..{span.at[s2, 'max']:%Y-%m}")
    return "; ".join(out) or "none"
unc["candidates"] = unc["cik"].map(cand)
unc["alpaca"] = unc["candidates"].map(alp)
unc = unc.sort_values("public_float", ascending=False)
unc.to_csv(OUT / "form25_uncovered_dcf_eligible.csv", index=False)
display(unc[["cik", "name", "filing_date", "public_float", "last_price", "candidates", "alpaca"]].head(25))

In [ ]:
# Exits among companies that were DCF-ready in the panel within 24 months before their last price
ready = panel[panel["fy4_fcf"].notna() & (panel["ttm_revenue"] > 0) & ~panel["is_financial"] & ~panel["is_blank_check"]
              & ~panel["mcap_suspect"] & (panel["ev"] > 0)][["cik", "month_end"]]
exr = ex.merge(ready, on="cik")
exr = exr[exr["month_end"] >= exr["last_date"] - pd.Timedelta(days=730)].drop_duplicates("cik")
t = exr.groupby([exr["last_date"].dt.year, "exit_type"]).size().unstack(fill_value=0)
t.to_csv(OUT / "exits_dcf_ready_by_year.csv"); display(t)

## Readout (fill in after the run)

- A. Apple check:
- B. DCF-ready universe per year and by EV bucket:
- C. Weakest items on the priced universe:
- D. Form 25 price coverage:
- E. Mapping suspects (share and examples):
- F. Exits, acquired vs other:
- Decisions for FV 03: universe filters (EV range, mega-cap exclusion), which items are required, delisting-return rule.